# Search and Order a Multi-Burst SBAS Stack of Interferograms



## Define a list of S1-burst geographic reference scenes for the SBAS searches

We suggest using a Geographic Search in ASF's Data Search [Vertex](https://hyp3-docs.asf.alaska.edu/guides/burst_insar_product_guide/#vertex) interface to identify a collection of reference bursts.

- The collection must not contain more than 15 bursts
- The bursts must be contiguous and from the same orbit pass
- There can be no holes in the collection of bursts
- They cannot intersect the antimeridian
- Bursts must be in the VV polarization
- Burst IDs in adjacent swaths must not have a difference greater than one
    - In simple terms:
      - Aim to build rectangular collections of bursts
      - L-shaped and T-shaped burst collections will likely fail to process

 ### This L-shaped collection of bursts is invalid:

 <img src="assets/invalid_multiburst_collection.png" alt="invalid L-shaped burst collection" style="width: 50%;"/>

 ### This rectangular collection of bursts is valid:
 
 <img src="assets/valid_multiburst_collection.png" alt="valid rectangular burst collection" style="width: 50%;"/>
 
Refer to the [Burst InSAR Product Guide](https://hyp3-docs.asf.alaska.edu/guides/burst_insar_product_guide/#considerations-for-selecting-input-bursts) for more guidance on selecting valid sets of Sentinel-1 bursts for multi-burst InSAR.


In [ ]:
from datetime import datetime

geo_ref_bursts = [
    "S1_071053_IW2_20210104T002658_VV_4542-BURST",
    "S1_071053_IW1_20210104T002657_VV_4542-BURST",
    "S1_071052_IW2_20210104T002655_VV_4542-BURST",
    "S1_071052_IW1_20210104T002654_VV_4542-BURST",
    "S1_071051_IW2_20210104T002652_VV_4542-BURST",
    "S1_071051_IW1_20210104T002651_VV_4542-BURST",
    "S1_071050_IW2_20210104T002650_VV_4542-BURST",
    "S1_071050_IW1_20210104T002649_VV_4542-BURST",
    "S1_071049_IW2_20210104T002647_VV_4542-BURST",
    "S1_071053_IW3_20210104T002659_VV_4542-BURST",
    "S1_071052_IW3_20210104T002656_VV_4542-BURST",
    "S1_071051_IW3_20210104T002653_VV_4542-BURST",
    "S1_071050_IW3_20210104T002651_VV_4542-BURST",
    "S1_071049_IW3_20210104T002648_VV_4542-BURST",
    "S1_071049_IW1_20210104T002646_VV_4542-BURST"
]

## Define a HyP3 project name and other parameters of the multiburst SBAS stack

In [ ]:
project_name = "multiburst" # Change this to something that makes sense for your project

start_date = datetime(2021, 1, 1)
end_date = datetime(2021, 1, 31)
temporal_baseline = 36
perpendicular_baseline = 400

## Perform an `asf_search.granule_search` on the S1 burst IDs of the georeferenced bursts

See this [notebook](https://github.com/asfadmin/Discovery-asf_search/blob/feature/stacking/examples/3-Granule_Search.ipynb) for more examples using `asf_search.granule_search`

In [ ]:
import asf_search as asf

results = asf.granule_search(geo_ref_bursts)
print(f"Found {len(results)} S1 burst SLCs")
results

## Perform an `asf_search.stack` search on each of the returned `S1BurstProduct`s

See this [notebook](https://github.com/asfadmin/Discovery-asf_search/blob/feature/stacking/examples/4-Baseline_Search.ipynb) for more example using `asf_search.stack` searches

In [ ]:
opts=asf.ASFSearchOptions(**{
    "start": start_date,
    "end": end_date,
})

stacks = [slc.stack(opts=opts) for slc in results]

## Create an SBAS stack of `asf_search.Pair`s for each of the stack searches

In [ ]:
from asf_search import Pair

pairs = [
    [
        pair
        for i, ref in enumerate(stack)
        for sec in stack[i+1:] # Prevent duplication. We only need Pair(a, b), not Pair(b, a)
        for pair in [Pair(ref, sec)]
        if pair.perpendicular_baseline is not None
        and pair.temporal_baseline is not None
        and pair.perpendicular_baseline <= perpendicular_baseline
        and 0 <= pair.temporal_baseline.days <= temporal_baseline
    ]
    for stack in stacks
]

pairs

## Transpose the individual stack search results into multiburst SBAS stacks

In [ ]:
transposed_pairs = list(map(list, zip(*pairs)))
transposed_pairs

## Convert the lists of multiburst `asf_search.Pair`s into corresponding lists of reference and secondary burst IDs for each multiburst interferogram

In [ ]:
multibursts_prepped_for_hyp3 = [
    [
        [pair.ref.properties["sceneName"] for pair in pair_list],
        [pair.sec.properties["sceneName"] for pair in pair_list],
    ]
    for pair_list in transposed_pairs
]
multibursts_prepped_for_hyp3

## Confirm the number of interferograms you are preparing to order, and the number of bursts they contain

In [ ]:
print(f"Preparing to order {len(multibursts_prepped_for_hyp3)} multiburst interferograms, each comprised of {len(multibursts_prepped_for_hyp3[0][0])} Sentinel-1 burst pairs.")

## Select whether you are accessing data from [HyP3 Basic](https://hyp3-docs.asf.alaska.edu/hyp3-docs/about/hyp3_basic/) or [HyP3+](https://hyp3-docs.asf.alaska.edu/hyp3-docs/about/hyp3_plus/)

In [ ]:
import ipywidgets as widgets
from IPython.display import display



hyp3_service_select = widgets.RadioButtons(
    options=[
        'HyP3 Basic',
        'HyP3+'
    ],
    description='Service:',
    disabled=False
)

display(hyp3_service_select)


## Authenticate with HyP3

In [ ]:
import hyp3_sdk as sdk

hyp3_select = hyp3_service_select.value

if "Basic" in hyp3_select:
    hyp3 = sdk.HyP3(prompt="password")
else:
    hyp3 = sdk.HyP3(api_url="https://hyp3-plus.asf.alaska.edu", prompt="password")

## Order the multiburst interferograms from HyP3

In [ ]:
multiburst_jobs = sdk.Batch()

for multiburst in multibursts_prepped_for_hyp3:
    multiburst_jobs += hyp3.submit_insar_isce_multi_burst_job(
        multiburst[0],
        multiburst[1],
        name=project_name,
        apply_water_mask=False,
        looks='20x4', # '20x4', '10x2', '5x1'
    )

## Watch the status of the multiburst jobs so you know when they have finished processing

In [ ]:
multiburst_jobs = hyp3.watch(multiburst_jobs)

## Search the completed HyP3 multiburst project

In [ ]:
user_id = input("Enter your NASA Earthdata Login username")

batch = sdk.Batch()
batch = hyp3.find_jobs(
    name=project_name, 
    job_type="INSAR_ISCE_MULTI_BURST", 
    user_id=user_id
)
batch

## Download the multiburst interferograms

In [ ]:
for job in batch:
    job.download_files()